# Impoliteness pilot — data prep

Design doc: `docs/superpowers/specs/2026-07-23-impoliteness-pilot-design.md`.

This section builds the paragraph-level dataset the pilot's sampling/LLM-scoring steps will
draw from: all paragraphs (regular speech + interjections) within a **±1-year window around
each state's own AfD entry date** — a per-state event window rather than one fixed calendar
year, so the pilot sample is anchored to the actual treatment timing per state instead of an
arbitrary shared year.

`DATA_ROOT` is set in two ways depending on environment:
- **Local**: read from `.env` in the project root (copy `.env.example` → `.env` and set your path)
- **Colab**: mount Google Drive in the cell below, then set `DATA_ROOT` to the Drive path


In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/My Drive/my_projects/M.A. Parliament/Code and Data/data"
    print("Status: Connected to Google Colab")
else:
    if "DATA_ROOT" not in os.environ:
        try:
            from dotenv import load_dotenv, find_dotenv
            load_dotenv(find_dotenv())
        except ImportError:
            pass
    DATA_ROOT = os.environ.get("DATA_ROOT", "")
    print("Status: Connected to a Local or Custom Kernel")

print("DATA_ROOT:", DATA_ROOT)
assert DATA_ROOT and os.path.isdir(DATA_ROOT), f"DATA_ROOT not set or missing: {DATA_ROOT!r}"


Status: Connected to a Local or Custom Kernel
DATA_ROOT: /Users/anna/Library/CloudStorage/GoogleDrive-anle.werner.01@gmail.com/My Drive/my_projects/M.A. Parliament/Code and Data/data


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW  = Path(DATA_ROOT) / "raw"
PROC = Path(DATA_ROOT) / "processed"
V3   = RAW / "stateparl_v3_parquet"


## Load paragraphs and the parsed nsc dataset

- `stateparl_v3_paragraphs.parquet` — every paragraph (speech + interjections), all affiliations.
- `nsc.parquet` — output of `preprocessing/nsc_rule_parser.py`: one row per parsed segment
  (multisegment rows already split, but `nsc_type`/`party_canonical` left pipe-joined and NOT
  further exploded on party/type). Using `nsc.parquet` directly, not the party×type-exploded
  `nsc_party_type.parquet`, avoids double-labeling the same span of text once per attributed
  party — see `docs/superpowers/specs/2026-07-27-nsc-pipeline-cleanup-design.md`. Only covers
  `affiliation == "nsc"` paragraphs. Non-interjection row types
  (`document_reference`/`Prozedural`/`misattributed`/`garbled`/`glocke`) are excluded below,
  before the join — see `preprocessing.nsc_rule_parser.NON_INTERJECTION_TYPES`.


In [ ]:
_cwd = Path(".").resolve()
_repo = _cwd if (_cwd / "preprocessing" / "nsc_rule_parser.py").exists() else _cwd.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from preprocessing.nsc_rule_parser import NON_INTERJECTION_TYPES

paragraphs = pd.read_parquet(V3 / "stateparl_v3_paragraphs.parquet")
paragraphs["date"] = pd.to_datetime(paragraphs["date"])
print(f"paragraphs: {paragraphs.shape[0]:,} rows x {paragraphs.shape[1]} cols, "
      f"{paragraphs['state'].nunique()} states")

nsc = pd.read_parquet(PROC / "nsc.parquet")
n_before = len(nsc)
nsc = nsc[~nsc["nsc_type"].isin(NON_INTERJECTION_TYPES)].copy()
print(f"nsc.parquet: {n_before:,} segments -> {len(nsc):,} after excluding "
      f"non-interjection row types ({n_before - len(nsc):,} dropped)")
print(f"  {nsc['paragraph_id'].nunique():,} distinct source paragraphs")


## AfD entry date per state

First legislative period (per state) with any `affiliation == "afd"` row, using that period's
own constitutive-session date (first sitting overall, not just the first AfD-affiliated row) —
the same corpus-derived method validated in `analysis/nsc_analysis.ipynb`'s `AFD_PRESENCE`
derivation (0–29 day gap vs. the first actual AfD row, vs. up to ~11 months of error from the
old entry-year-only proxy). This pilot only needs the single first-entry "shock" date per
state, not the full presence/exit history `AFD_PRESENCE` tracks — Bremen and Schleswig-Holstein
later exited, but their *entry* window is still the relevant one for this pilot's ±1yr framing.


In [16]:
AFD_ENTRY = (pd.read_csv(PROC / "afd_entry_dates.csv", parse_dates=["entry_date"])
                    .set_index("state")["entry_date"]
                    .dt.date
                    .to_dict())

print(f"States with an AfD entry date: {len(AFD_ENTRY)} of {paragraphs['state'].nunique()}")
for state, entry in sorted(AFD_ENTRY.items()):
    print(f"  {state}: {entry}")

States with an AfD entry date: 16 of 16
  bb: 2014-10-08
  be: 2016-10-27
  bw: 2016-05-11
  by: 2018-11-05
  hb: 2015-07-01
  he: 2019-01-18
  hh: 2015-09-16
  mv: 2016-10-04
  ni: 2017-11-14
  nw: 2017-06-27
  rp: 2016-05-18
  sh: 2017-06-06
  sl: 2017-04-25
  sn: 2014-09-29
  st: 2016-04-12
  th: 2014-10-14


## Filter to ±1 year around each state\'s AfD entry


In [17]:
WINDOW = pd.DateOffset(years=1)
ENTRY_WINDOW = {s: (d - WINDOW, d + WINDOW) for s, d in AFD_ENTRY.items()}

mask = pd.Series(False, index=paragraphs.index)
for state, (start, end) in ENTRY_WINDOW.items():
    mask |= (paragraphs["state"] == state) & paragraphs["date"].between(start, end)

para_window = paragraphs[mask].copy()
print(f"Filtered: {len(para_window):,} of {len(paragraphs):,} paragraphs "
      f"({len(para_window)/len(paragraphs)*100:.1f}%), {para_window['state'].nunique()} states")
print()
print(para_window.groupby("state").size().sort_values(ascending=False).to_string())


Filtered: 1,223,597 of 16,078,467 paragraphs (7.6%), 16 states

state
nw    112336
ni    107797
he     99050
mv     98940
bw     91776
th     87036
sh     82839
st     79069
by     72397
sn     67972
rp     63672
hh     62186
be     61660
hb     60729
bb     49326
sl     26812


## Join filtered `nsc` segments onto the windowed paragraphs

Left join on `paragraph_id` only, using `nsc.parquet` directly (not the party×type-exploded
`nsc_party_type.parquet` from the earlier design) — classification needs one row per real
segment of text, not one row per (segment, type, party) combination, or the same span of text
would get sent to the LLM and labeled multiple times whenever it's attributed to more than one
party. `nsc` columns that already exist identically in `paragraphs` are dropped first, same
redundant set as before: `protocol_id`/`speech_id`/`state`/`period`/`nth`/`date`/
`protocol_position` (already in `paragraphs`), `raw_row` (duplicates `paragraphs`' own `content`
verbatim).

Non-`nsc` paragraphs (regular speech, government, president, ...) get exactly one output row
with all `nsc` columns `NaN` — no match. `nsc`-affiliated paragraphs (already filtered to real
interjection types above) fan out to one row per matching segment — one row for the common
single-segment case, several for multi-segment source rows.


In [ ]:
# inspect nsc
nsc.head(100)

In [19]:
# inspect paragraphs
paragraphs.head()

,paragraph_id,protocol_id,state,period,nth,date,speech_id,protocol_position,segment_position,page,speaker_paragraph,mandate_id,affiliation,content
0,1,bb_3_10,bb,3,10,2000-02-24,NaN,1,2,4,Knoblich,bb_3_pre_knoblich,pre,Werte Kolleginnen und Kollegen! Ich begrüße Si...
1,2,bb_3_10,bb,3,10,2000-02-24,NaN,2,3,4,,,nsc,(Allgemeiner Beifall)
2,3,bb_3_10,bb,3,10,2000-02-24,NaN,3,4,4,Knoblich,bb_3_pre_knoblich,pre,Nicht weniger herzlich begrüße ich unsere Stam...
3,4,bb_3_10,bb,3,10,2000-02-24,NaN,4,5,4,Knoblich,bb_3_pre_knoblich,pre,Mit der Einladung ist Ihnen der Vorschlag zur ...
4,5,bb_3_10,bb,3,10,2000-02-24,NaN,5,6,4,Knoblich,bb_3_pre_knoblich,pre,"Ich darf darauf hinweisen, dass die DVU-Frakti..."


raw row == content

date is with tim ein paragraphs and without in nsc_llm

one protocol_position from paragraphs can have multiple nsc from nsc_llm 
 why is speech id NA in paragraphs.head() ??

 okay beifall bei der SPD und der CDU is exploded. i reconsidered, I do not want to explode this for the classification df, since the same segment is then labelled twice. but i need this option of exploding for the time series and the aggregate df later. so exploding on unique parties will happen later. but what needs to get exploded right away is multisegment nscs. So I want only one dataframe that is built from the filtered nsc only df. and that is the one where I have multisegments exploded but party not yet. do you know what i mean?  This making an own row for every multisegment nsc should be the very first thing to happen. and from there, we clean the rest of the nsc. What do you think? What dataframe do i need for the rest of the project? How do I handle this with paragraph id? should i not filter to the nsc data frame at all and work directly on paragraphs parqeut? and i want nsc to become another binary. so that it is clear wether a paragraph was an nsc or not, and affiliation is filled in with the from nsc derived content correct speaker affiliation. important also for classificaton: clean tehe () brackets around nsc content or not? Also reassess the garbled and mislablled and noise and proceduarl categories. which of them can be extracted cleanly and which not, which need the distinction and which do not. if it hinders the most important extractions, namely party and nsc_type, this is more important than distinguishing for more than actually needed. or should i use ML method rather than regex / rule parsing? 

In [ ]:
_REDUNDANT = ["protocol_id", "speech_id", "state", "period", "nth", "date",
              "protocol_position", "raw_row"]
nsc_slim = nsc.drop(columns=_REDUNDANT)

merged = para_window.merge(nsc_slim, on="paragraph_id", how="left")

print(f"para_window: {len(para_window):,} rows  ->  merged: {len(merged):,} rows")
print(f"  (+{len(merged) - len(para_window):,} from multi-segment nsc paragraphs fanning out)")
print()

n_nsc_paragraphs = (para_window["affiliation"] == "nsc").sum()
n_matched_paragraphs = merged.loc[merged["nsc_type"].notna(), "paragraph_id"].nunique()
print(f"nsc-affiliated paragraphs in window: {n_nsc_paragraphs:,}")
print(f"  of which matched >=1 real-interjection segment: {n_matched_paragraphs:,}")
print(f"  (gap = paragraphs where every segment was a non-interjection row type -- "
      f"document_reference/Prozedural/misattributed/garbled/glocke -- already excluded above)")
print()

non_nsc = merged[merged["affiliation"] != "nsc"]
print(f"Non-nsc rows: {len(non_nsc):,}; any duplicated paragraph_id (should be False)? "
      f"{non_nsc['paragraph_id'].duplicated().any()}")
print(f"Non-nsc rows with non-null nsc_type (should be 0): {non_nsc['nsc_type'].notna().sum()}")


## Sanity check: preview a joined nsc row and a joined non-nsc row


In [ ]:
cols = ["paragraph_id", "state", "period", "date", "affiliation", "content",
        "nsc_type", "party_canonical", "affiliation_derived", "speaker_name", "content_text"]

print("-- example nsc paragraph (fanned out) --")
example_pid = merged.loc[merged["nsc_type"].notna(), "paragraph_id"].iloc[0]
print(merged[merged["paragraph_id"] == example_pid][cols].to_string(index=False))

print()
print("-- example non-nsc paragraph (single row, nsc columns NaN) --")
print(merged[merged["affiliation"] != "nsc"][cols].head(2).to_string(index=False))


## LLM scoring — zero-shot impoliteness classification

Sample paragraphs from `merged` and classify each with a local Qwen3-14B model via Ollama.
See `docs/superpowers/specs/2026-07-23-impoliteness-pilot-design.md` (Part 2) for the design
rationale: binary output, one call per paragraph, everything seeded.

In [ ]:
import subprocess

MODEL = "qwen3:14b-q4_K_M"

result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if MODEL not in result.stdout:
    print(f"Pulling {MODEL} (this downloads ~30GB, may take a while)...")
    subprocess.run(["ollama", "pull", MODEL], check=True)
else:
    print(f"{MODEL} already pulled.")

In [ ]:
SEED = 42
N_SAMPLE = 300

is_nsc_row = merged["affiliation"] == "nsc"
merged["text_to_classify"] = merged["content"]
merged.loc[is_nsc_row, "text_to_classify"] = merged.loc[is_nsc_row, "content_text"]

# nsc rows with blank content_text (parser didn't extract usable text for that segment) are
# dropped from the classifiable set here -- not sent to the LLM as an empty string, and not
# falling back to raw_segment/raw_row either (party attribution still visible there -- see
# Decision 6, docs/superpowers/specs/2026-07-27-nsc-pipeline-cleanup-design.md). This is a
# known coverage gap (~11.7% of Zwischenruf rows), not a design choice fixed here.
#
# Two distinct blank cases, both need excluding: (1) content_text == "" -- the segment
# matched but the parser extracted nothing; (2) text_to_classify is None -- the paragraph's
# entire nsc.parquet entry was excluded above (every segment was a non-interjection
# category), so the left join produced no match at all. affiliation still says "nsc" (from
# paragraphs, independent of the join outcome), but there's no content_text to read. A plain
# != "" check treats None as "not blank" (None != "" is True in pandas), so .notna() must
# be checked explicitly too, or case (2) silently leaks an empty prompt into the sample.
classifiable = merged[
    ~is_nsc_row | (merged["text_to_classify"].notna() & (merged["text_to_classify"] != ""))
].copy()
print(f"Dropped {len(merged) - len(classifiable):,} nsc rows with blank content_text "
      f"(no usable extracted text)")

# One row per physical paragraph/segment -- nsc.parquet (unlike the old nsc_llm join) is
# not exploded on party, so this dedup is now a defensive safety net rather than undoing a
# party cross-product, but kept for robustness.
dedup = classifiable.drop_duplicates(subset=["paragraph_id", "segment_idx"], keep="first")
sample = dedup.sample(n=N_SAMPLE, random_state=SEED).reset_index(drop=True)

print(f"Deduplicated {len(classifiable):,} rows -> {len(dedup):,} unique paragraph/segment texts")
print(f"Sampled {len(sample):,} rows for LLM scoring")
sample[["paragraph_id", "state", "period", "affiliation", "text_to_classify"]].head()

In [10]:
import ollama

from impoliteness_lib import build_prompt, parse_response

_test_response = ollama.chat(
    model=MODEL,
    messages=build_prompt("Das ist doch eine Frechheit, Sie Lügner!"),
    think=False,
    format="json",
    options={"temperature": 0, "seed": SEED},
)
print(_test_response["message"]["content"])
print(parse_response(_test_response["message"]["content"]))

{"impolite": true, "reason": "Verwendung von Beleidigungen und Anschuldigungen"}
{'impolite': True, 'reason': 'Verwendung von Beleidigungen und Anschuldigungen', 'raw_output': '{"impolite": true, "reason": "Verwendung von Beleidigungen und Anschuldigungen"}'}


In [ ]:
records = []
for i, row in sample.iterrows():
    response = ollama.chat(
        model=MODEL,
        messages=build_prompt(row["text_to_classify"]),
        think=False,
        format="json",
        options={"temperature": 0, "seed": SEED},
    )
    parsed = parse_response(response["message"]["content"])
    records.append({
        "paragraph_id": row["paragraph_id"],
        "state": row["state"],
        "period": row["period"],
        "date": row["date"],
        "affiliation": row["affiliation"],
        "content": row["text_to_classify"],
        "impolite": parsed["impolite"],
        "reason": parsed["reason"],
        "model_name": MODEL,
    })
    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(sample)} scored")

predictions = pd.DataFrame(records)
n_unparsed = predictions["impolite"].isna().sum()
print(f"Scored {len(predictions):,} paragraphs; {n_unparsed} unparseable responses")

In [ ]:
out_path = Path(DATA_ROOT) / "measurement" / "impoliteness_pilot_predictions.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")

In [ ]:
rate = predictions["impolite"].mean()
print(f"Predicted impoliteness rate: {rate:.1%} "
      f"({predictions['impolite'].sum():.0f} of {predictions['impolite'].notna().sum()} "
      f"parseable predictions)")
print()
print("By state:")
print(predictions.groupby("state")["impolite"].mean().sort_values(ascending=False)
      .apply(lambda x: f"{x:.1%}").to_string())

In [ ]:
print("-- Examples predicted impolite --")
for _, row in predictions[predictions["impolite"] == True].head(10).iterrows():
    print(f"[{row['state']} {row['period']}] {row['content']}")
    print(f"  -> reason: {row['reason']}")
    print()

In [ ]:
gold = pd.read_csv("../labelling/annotations_output.csv", dtype={"para_id": str})
gold_neutral = gold[gold["politeness"] == "neutral"]

check = predictions.merge(
    gold_neutral[["para_id", "politeness"]],
    left_on="paragraph_id", right_on="para_id", how="inner",
)
print(f"Overlap with gold neutral rows: {len(check)}")
if len(check):
    print(check[["paragraph_id", "impolite", "content"]].to_string(index=False))
    n_flagged = (check["impolite"] == True).sum()
    print(f"\n{n_flagged} of {len(check)} gold-neutral rows flagged impolite by the model "
          f"(should be 0 or very low; if not, revisit the prompt)")
else:
    print("No overlap between this sample and the 36 gold-labeled rows - expected, since "
          "they're separate random draws from a much larger pool. This check only becomes "
          "meaningful once the gold set is larger or the sample size increases.")